In [1]:
!pip install pinecone cohere sentence-transformers tqdm
!pip install langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 115.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 128.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 104.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# 1. Install required packages (run this once)
!pip install pinecone cohere sentence-transformers tqdm

# 2. Import libraries
import re
from tqdm.auto import tqdm
from uuid import uuid4
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec
import cohere
import numpy as np

# 3. Load and clean the text
from langchain_community.document_loaders import GutenbergLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

try:
    loader = GutenbergLoader("https://www.gutenberg.org/cache/epub/49513/pg49513.txt")
    data = loader.load()
    text = data[0].page_content
    text = re.sub(r"\*+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = text[16398:-64871]  # Trim document
except Exception as e:
    print(f"Error loading text: {e}")

# 4. Split text into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
texts = text_splitter.split_text(text)

# 5. Generate embeddings using SentenceTransformer
embedding_model = SentenceTransformer('all-mpnet-base-v2')
embeddings = embedding_model.encode(texts, normalize_embeddings=True)
print("Embedding shape:", embeddings[0].shape)

# 6. Initialize Pinecone
pc = Pinecone(api_key="pcsk_6Tb3xu_6xdh4JrKfsi2uWjGFfqw7XT5Q1C1DyJkaJNk4Ke8TVcRi9iHPDEbkECniUVX6XW")

# pc.delete_index("herbal-knowledge-base")
pc.delete_index("herbal-knowledge-base")

index_name = "herbal-knowledge-base"
embedding_dim = 768

# 7. Create Pinecone index if needed
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=embedding_dim,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)

# 8. Upsert embeddings into Pinecone
batch_size = 100
for i in range(0, len(texts), batch_size):
    batch_texts = texts[i:i+batch_size]
    batch_embeddings = embeddings[i:i+batch_size]
    ids = [str(uuid4()) for _ in batch_texts]
    metadata = [{"text": text} for text in batch_texts]
    vectors = list(zip(ids, [v.tolist() for v in batch_embeddings], metadata))
    index.upsert(vectors)
    print(f"Upserted {i}–{i+len(batch_texts)}")

# 9. Retrieval function using Pinecone
def retrieve_relevant_chunks(query, embedding_model, pinecone_index, top_k=5):
    query_embedding = embedding_model.encode([query], normalize_embeddings=True)[0]
    results = pinecone_index.query(vector=query_embedding.tolist(), top_k=top_k, include_metadata=True)
    return [(match['metadata']['text'], match['score']) for match in results['matches']]

# 10. Response generation using Cohere
cohere_api_key = "aRoTuluolENeFP2nvcO09ujU3nESZDYvtAja3hv0"
co = cohere.Client(cohere_api_key)

def generate_response(query, embedding_model, pinecone_index, top_k=5):
    relevant_chunks = retrieve_relevant_chunks(query, embedding_model, pinecone_index, top_k)
    if not relevant_chunks:
        return "No relevant information found."

    context = "\n".join([chunk for chunk, _ in relevant_chunks])
    message = f"You are a helpful assistant. Answer this question based ONLY on the following information:\n\n{context}\n\nQuestion: {query}"

    try:
        response = co.chat(
            message=message,
            model="command-xlarge-nightly",
            preamble="Answer based only on the given context and find answers from the context. Do not hallucinate.",
            max_tokens=200,
            temperature=0.7
        )
        return response.text
    except Exception as e:
        return f"Error from Cohere: {e}"

# 11. Chatbot loop
# def chatbot():
#     print("Welcome to the Herbal Medicines Chatbot! Type 'exit' to quit.")
#     while True:
#         query = input("\nEnter your question: ")
#         if query.lower() == "exit":
#             print("Goodbye!")
#             break
#         answer = generate_response(query, embedding_model, index)
#         print("\n🤖 Chatbot Response:")
#         print(answer)

# # 12. Run the chatbot
# if __name__ == "__main__":
#     chatbot()

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.4k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding shape: (768,)
Upserted 0–100
Upserted 100–200
Upserted 200–300
Upserted 300–400
Upserted 400–500
Upserted 500–600
Upserted 600–700
Upserted 700–800
Upserted 800–900
Upserted 900–1000
Upserted 1000–1100


In [ ]:
!pip install --upgrade gradio

In [ ]:
import gradio as gr

def gradio_chat(query):
    if query.lower() == "exit":
        return "Goodbye!"
    answer = generate_response(query, embedding_model, index)
    return answer

# Launch the Gradio interface
iface = gr.Interface(
    fn=gradio_chat,
    inputs=gr.Textbox(lines=3, placeholder="Ask me about herbal medicine...🍃"),
    outputs="text",
    title="Herbal Medicine Chatbot🌿 ",
    description="Welcome to your Herbal solution! Ask me about any herb and I will give you remedies.",
    theme=gr.themes.Soft(primary_hue="green"),
    allow_flagging='never'
)

iface.launch()

In [ ]:
test_data = [
    {
        "question": "ALKANET?",
        "reference": """BESIDES the common name, it is called Orchanet, and Spanish Bugloss,and by apothecaries, Enchusa.
                        _Descript._] Of the many sorts of this herb, there is but one known to
                        grow commonly in this nation; of which one take this description: It
                        hath a great and thick root, of a reddish colour, long, narrow, hairy
                        leaves, green like the leaves of Bugloss, which lie very thick upon the
                        ground; the stalks rise up compassed round about, thick with leaves,
                        which are less and narrower than the former; they are tender, and
                        slender, the flowers are hollow, small, and of a reddish colour.
                        _Place._] It grows in Kent near Rochester, and in many places in the
                        West Country, both in Devonshire and Cornwall._Time._] They flower in July and the beginning of August, and the
                        seed is ripe soon after, but the root is in its prime, as carrots and
                        parsnips are, before the herb runs up to stalk._Government and virtues._] It is an herb under the dominion of Venus,
                        and indeed one of her darlings, though somewhat hard to come by. It
                        helps old ulcers, hot inflammations, burnings by common fire, and St.
                        Anthony’s fire, by antipathy to Mars; for these uses, your best way is
                        to make it into an ointment; also, if you make a vinegar of it, as you
                        make vinegar of roses, it helps the morphew and leprosy; if you apply
                        the herb to the privities, it draws forth the dead child. It helps the
                        yellow jaundice, spleen, and gravel in the kidneys. Dioscorides saith
                        it helps such as are bitten by a venomous beast, whether it be taken
                        inwardly, or applied to the wound; nay, he saith further, if any one
                        that hath newly eaten it, do but spit into the mouth of a serpent, the
                        serpent instantly dies. It stays the flux of the belly, kills worms,
                        helps the fits of the mother. Its decoction made in wine, and drank,
                        strengthens the back, and eases the pains thereof: It helps bruises
                        and falls, and is as gallant a remedy to drive out the small pox and
                        measles as any is; an ointment made of it, is excellent for green
                        wounds, pricks or thrusts."""
    },
    {
        "question": "ALL-HEAL.",
        "reference": """IT is called All-heal, Hercules’s All-heal, and Hercules’s Woundwort,
                        because it is supposed that Hercules learned the herb and its virtues
                        from Chiron, when he learned physic of him. Some call it Panay, and
                        others Opopane-wort._Descript._] Its root is long, thick, and exceeding full of juice, of
                        a hot and biting taste, the leaves are great and large, and winged
                        almost like ash-tree leaves, but that they are something hairy, each
                        leaf consisting of five or six pair of such wings set one against the
                        other upon foot-stalks, broad below, but narrow towards the end; one
                        of the leaves is a little deeper at the bottom than the other, of a
                        fair yellowish fresh green colour: they are of a bitterish taste,
                        being chewed in the mouth; from among these rises up a stalk, green in
                        colour, round in form, great and strong in magnitude, five or six feet
                        in altitude, with many joints, and some leaves thereat; towards the top
                        come forth umbels of small yellow flowers, after which are passed away,
                        you may find whitish, yellow, short, flat seeds, bitter also in taste.
                        _Place._] Having given you a description of the herb from bottom to
                        top, give me leave to tell you, that there are other herbs called by
                        this name; but because they are strangers in England, I give only the
                        description of this, which is easily to be had in the gardens of divers
                        places._Time._] Although Gerrard saith, that they flower from the beginning
                        of May to the end of December, experience teaches them that keep it in
                        their gardens, that it flowers not till the latter end of the summer,
                        and sheds its seeds presently after._Government and virtues._] It is under the dominion of Mars, hot,
                        biting, and choleric; and remedies what evils Mars inflicts the body
                        of man with, by sympathy, as vipers’ flesh attracts poison, and
                        the loadstone iron. It kills the worms, helps the gout, cramp, and
                        convulsions, provokes urine, and helps all joint-aches. It helps all
                        cold griefs of the head, the vertigo, falling-sickness, the lethargy,
                        the wind cholic, obstructions of the liver and spleen, stone in the
                        kidneys and bladder. It provokes the terms, expels the dead birth:
                        it is excellent good for the griefs of the sinews, itch, stone, and
                        tooth-ache, the biting of mad dogs and venomous beasts, and purges
                        choler very gently."""
    }
]

In [ ]:
predictions = []
references = []

for item in test_data:
    prediction = generate_response(item["question"], embedding_model, index)
    predictions.append(prediction)
    references.append(item["reference"])

In [ ]:
!pip install evaluate bert-score
!pip install ipywidgets

In [ ]:
# BERTScore
from bert_score import score
P, R, F1 = score(predictions, references, lang="en", verbose=True)
print(f"BERTScore - Precision: {P.mean().item():.4f}, Recall: {R.mean().item():.4f}, F1: {F1.mean().item():.4f}")